In [1]:
from tensorflow.keras.layers import Flatten, Dense, Conv2D, MaxPooling2D, Dropout
from tensorflow.keras.applications.resnet_v2 import ResNet152V2
from tensorflow.keras import applications
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow import device
from matplotlib import pyplot
from PIL import ImageFile
import sys
from keras_preprocessing.image import ImageDataGenerator
import os
import tensorflow as tf

In [2]:


# def min_image_count():
#     root = "./Train/"
#     datasets = ["train/", "test/"]
#     features = ["heel_strike","other","toe_off"]

#     train_count = None
#     test_count = None

#     for i, dataset in enumerate(datasets):

#         val = 1000
#         for feature in features:
#             dir_path = root + dataset + feature
#             files = os.listdir(dir_path)
#             image_count = len(files)
#             print(dataset, feature, image_count)

#             if image_count < val:
#                 val = image_count - 1

#         if i == 0:
#             train_count = val
#         elif i == 1:
#             test_count = val

#     return [train_count, test_count]




# min_image_count()




def get_generators(shuffle):
    train_datagen = ImageDataGenerator(rescale=1.0 / 255.0, validation_split=0.2)
    test_datagen = ImageDataGenerator(rescale=1.0 / 255.0, validation_split=0.2)

    if shuffle:
    
        train_datagen = ImageDataGenerator(rescale=1.0/255.0, width_shift_range=0.1,
                                           height_shift_range=0.1, horizontal_flip=True, validation_split=0.2)

    train_it = train_datagen.flow_from_directory("./Train/", class_mode="categorical",
                                                 batch_size=16, target_size=(128, 128), shuffle=shuffle, subset='training')

    test_it = test_datagen.flow_from_directory("./Train/", class_mode="categorical", batch_size=32,
                                               target_size=(128, 128), shuffle=shuffle, subset='validation')

    return train_it, test_it



# ResNet152

In [3]:


optimizer = tf.keras.optimizers.Nadam(learning_rate=0.001, beta_1=0.9,beta_2=0.999)

model = ResNet152V2(include_top=False, input_shape=(128, 128, 3))
for layer in model.layers:
    layer.trainable = False

dropout = Dropout(0.1)(model.layers[-1].output)
flat = Flatten()(dropout)
dense = Dense(128, activation="relu", kernel_initializer="he_uniform")(flat)
dropout = Dropout(0.05)(dense)
dense = Dense(64)(dropout)
dropout = Dropout(0.05)(dense)
dense = Dense(48)(dropout)
dropout = Dropout(0.05)(dense)
dense = Dense(36)(dropout)
dropout = Dropout(0.05)(dense)
dense = Dense(24)(dropout)
dense = Dense(12)(dense)
output = Dense(3, activation="softmax")(dense)

model = Model(inputs=model.inputs, outputs=output)
model.compile(optimizer=optimizer, loss="categorical_crossentropy", metrics="accuracy")





ImageFile.LOAD_TRUNCATED_IMAGES = True



es = EarlyStopping(monitor="val_accuracy", min_delta=0.005, verbose=1, patience=2, restore_best_weights=True)

train_it, test_it = get_generators(True)

history = model.fit(train_it, steps_per_epoch=len(train_it), callbacks=[es],
                    validation_data=test_it, validation_steps=len(test_it), epochs=10, verbose=1)

_, acc = model.evaluate(test_it, steps=len(test_it), verbose=0)

model.save("./models/ResNet152V2.h5")
print("Finished training")
print('> %.3f' % (acc * 100.0))


Found 2246 images belonging to 3 classes.
Found 560 images belonging to 3 classes.
Epoch 1/10
71/71 [==============================] - 887s 12s/step - loss: 0.9661 - accuracy: 0.7431 - val_loss: 0.5548 - val_accuracy: 0.7714
Epoch 2/10
71/71 [==============================] - 62s 868ms/step - loss: 0.5764 - accuracy: 0.7698 - val_loss: 0.5569 - val_accuracy: 0.7750
Epoch 3/10
71/71 [==============================] - 62s 872ms/step - loss: 0.5565 - accuracy: 0.7711 - val_loss: 0.5273 - val_accuracy: 0.7750
Epoch 3: early stopping
Finished training
> 77.143


# InceptionV3

In [4]:

optimizer = tf.keras.optimizers.Nadam(learning_rate=0.001, beta_1=0.9,beta_2=0.999)
model = applications.inception_v3.InceptionV3(include_top=False, input_shape=(128, 128, 3))
for layer in model.layers:
    layer.trainable = False

dropout = Dropout(0.1)(model.layers[-1].output)
flat = Flatten()(dropout)
dense = Dense(128, activation="relu", kernel_initializer="he_uniform")(flat)
dropout = Dropout(0.05)(dense)
dense = Dense(64)(dropout)
dropout = Dropout(0.05)(dense)
dense = Dense(48)(dropout)
dropout = Dropout(0.05)(dense)
dense = Dense(36)(dropout)
dropout = Dropout(0.05)(dense)
dense = Dense(24)(dropout)
dense = Dense(12)(dense)
output = Dense(3, activation="softmax")(dense)

model = Model(inputs=model.inputs, outputs=output)
model.compile(optimizer=optimizer, loss="categorical_crossentropy", metrics="accuracy")




ImageFile.LOAD_TRUNCATED_IMAGES = True


es = EarlyStopping(monitor="val_accuracy", min_delta=0.005, verbose=1, patience=2, restore_best_weights=True)

train_it, test_it = get_generators(True)

history = model.fit(train_it, steps_per_epoch=len(train_it), callbacks=[es],
                    validation_data=test_it, validation_steps=len(test_it), epochs=10, verbose=1)

_, acc = model.evaluate(test_it, steps=len(test_it), verbose=0)

model.save("./models/inception_v3.h5")
print("Finished training")
print('> %.3f' % (acc * 100.0))



87910968/87910968 [==============================] - 2s 0us/step
Found 2246 images belonging to 3 classes.
Found 560 images belonging to 3 classes.
Epoch 1/10
71/71 [==============================] - 25s 318ms/step - loss: 0.8551 - accuracy: 0.7551 - val_loss: 0.6273 - val_accuracy: 0.7750
Epoch 2/10
71/71 [==============================] - 21s 303ms/step - loss: 0.6026 - accuracy: 0.7671 - val_loss: 0.5619 - val_accuracy: 0.7750
Epoch 3/10
71/71 [==============================] - 22s 307ms/step - loss: 0.5770 - accuracy: 0.7703 - val_loss: 0.5724 - val_accuracy: 0.7750
Epoch 3: early stopping
Finished training
> 77.500


# InceptionResNetV2

In [5]:


optimizer = tf.keras.optimizers.Nadam(learning_rate=0.001, beta_1=0.9,beta_2=0.999)
model = applications.InceptionResNetV2(include_top=False, input_shape=(128, 128, 3))
for layer in model.layers:
    layer.trainable = False

dropout = Dropout(0.1)(model.layers[-1].output)
flat = Flatten()(dropout)
dense = Dense(128, activation="relu", kernel_initializer="he_uniform")(flat)
dropout = Dropout(0.05)(dense)
dense = Dense(64)(dropout)
dropout = Dropout(0.05)(dense)
dense = Dense(48)(dropout)
dropout = Dropout(0.05)(dense)
dense = Dense(36)(dropout)
dropout = Dropout(0.05)(dense)
dense = Dense(24)(dropout)
dense = Dense(12)(dense)
output = Dense(3, activation="softmax")(dense)

model = Model(inputs=model.inputs, outputs=output)
model.compile(optimizer=optimizer, loss="categorical_crossentropy", metrics="accuracy")




ImageFile.LOAD_TRUNCATED_IMAGES = True


es = EarlyStopping(monitor="val_accuracy", min_delta=0.005, verbose=1, patience=2, restore_best_weights=True)

train_it, test_it = get_generators(True)

history = model.fit(train_it, steps_per_epoch=len(train_it), callbacks=[es],
                    validation_data=test_it, validation_steps=len(test_it), epochs=10, verbose=1)

_, acc = model.evaluate(test_it, steps=len(test_it), verbose=0)

model.save("./models/InceptionResNetV2.h5")
print("Finished training")
print('> %.3f' % (acc * 100.0))


219055592/219055592 [==============================] - 4s 0us/step
Found 2246 images belonging to 3 classes.
Found 560 images belonging to 3 classes.
Epoch 1/10
71/71 [==============================] - 65s 828ms/step - loss: 0.7604 - accuracy: 0.7484 - val_loss: 0.6169 - val_accuracy: 0.7750
Epoch 2/10
71/71 [==============================] - 42s 596ms/step - loss: 0.6029 - accuracy: 0.7711 - val_loss: 0.5842 - val_accuracy: 0.7750
Epoch 3/10
71/71 [==============================] - 40s 568ms/step - loss: 0.5955 - accuracy: 0.7707 - val_loss: 0.5851 - val_accuracy: 0.7750
Epoch 3: early stopping
Finished training
> 77.500


# Xception

In [6]:

optimizer = tf.keras.optimizers.Nadam(learning_rate=0.001, beta_1=0.9,beta_2=0.999)
model = applications.Xception(include_top=False, input_shape=(128, 128, 3))
for layer in model.layers:
    layer.trainable = False

dropout = Dropout(0.1)(model.layers[-1].output)
flat = Flatten()(dropout)
dense = Dense(128, activation="relu", kernel_initializer="he_uniform")(flat)
dropout = Dropout(0.05)(dense)
dense = Dense(64)(dropout)
dropout = Dropout(0.05)(dense)
dense = Dense(48)(dropout)
dropout = Dropout(0.05)(dense)
dense = Dense(36)(dropout)
dropout = Dropout(0.05)(dense)
dense = Dense(24)(dropout)
dense = Dense(12)(dense)
output = Dense(3, activation="softmax")(dense)

model = Model(inputs=model.inputs, outputs=output)
model.compile(optimizer=optimizer, loss="categorical_crossentropy", metrics="accuracy")




ImageFile.LOAD_TRUNCATED_IMAGES = True


es = EarlyStopping(monitor="val_accuracy", min_delta=0.005, verbose=1, patience=2, restore_best_weights=True)

train_it, test_it = get_generators(True)

history = model.fit(train_it, steps_per_epoch=len(train_it), callbacks=[es],
                    validation_data=test_it, validation_steps=len(test_it), epochs=10, verbose=1)

_, acc = model.evaluate(test_it, steps=len(test_it), verbose=0)

model.save("./models/Xception.h5")
print("Finished training")
print('> %.3f' % (acc * 100.0))



83683744/83683744 [==============================] - 1s 0us/step
Found 2246 images belonging to 3 classes.
Found 560 images belonging to 3 classes.
Epoch 1/10
71/71 [==============================] - 42s 563ms/step - loss: 0.8244 - accuracy: 0.7422 - val_loss: 0.5298 - val_accuracy: 0.7821
Epoch 2/10
71/71 [==============================] - 32s 456ms/step - loss: 0.5585 - accuracy: 0.7734 - val_loss: 0.5235 - val_accuracy: 0.7821
Epoch 3/10
71/71 [==============================] - 32s 455ms/step - loss: 0.5548 - accuracy: 0.7676 - val_loss: 0.5239 - val_accuracy: 0.7768
Epoch 3: early stopping
Finished training
> 78.214
